# Lab 10: Wikipedia's APIs

*INFO 4617/5617 · Week 10 · Chapter 10, [Introduction to APIs: Wikipedia](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-10-wikipedia.html)*

Today we code together. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in.

By the end you'll have asked two of Wikipedia's APIs for one article's whole public record: its basic facts, every edit since 2003, its daily readers, who those readers are, and what the article connects to. Then you'll do the same with a client library in three lines.

**The article:** [University of Colorado Boulder](https://en.wikipedia.org/wiki/University_of_Colorado_Boulder), the chapter's case. Once a step works, try it on an article you care about.

| Step | What we build | Minutes |
|---|---|---|
| 0 | Setup: libraries and a User-Agent | 5 |
| 1–2 | One request, and reading its JSON | 10 |
| 3–4 | Every revision, one batch at a time, and a chart of them | 20 |
| 5–7 | Pageviews from a second API, the biggest spike, and who is reading | 20 |
| 8 | One function for links, categories, and languages | 10 |
| 9 | The same, with mwclient | 5 |
| 10 | What did we learn? | 5 |

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent.

## Step 0 — Set up (5 minutes)

Import `requests`, `pandas`, and `matplotlib`. Then define the User-Agent every request will send. Wikimedia's [User-Agent policy](https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy) asks every client to say who it is and how to reach its operator, in this form: `<client name>/<version> (<contact information>) <library name>/<version>`. Keep the first two parts in `CLIENT`, because Step 9's library adds its own name, and add `requests` and its version for `HEADERS`. A client the [rate limits](https://www.mediawiki.org/wiki/Wikimedia_APIs/Rate_limits) can identify only by its network address gets 10 requests a minute, shared with everyone on that network, which is everyone in this room. One with a User-Agent like this gets 200, and if it still gets a 429, the response's `Retry-After` header says how many seconds to wait.

In [ ]:
# Import requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Define CLIENT as "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# Define HEADERS: a dict whose "User-Agent" is CLIENT, a space, then "python-requests/" and requests.__version__
# Define API, the Action API's address, and ARTICLE, the article's title
# Your code here

## Step 1 — One request (5 minutes)

Every request to the MediaWiki [Action API](https://www.mediawiki.org/wiki/API:Action_API) has four parts: the base URL, an `action` (what to do), parameters that narrow it, and a `format`. Build the parameters as a dictionary, and `requests` turns them into the URL. Ask for `prop=info`, the article's basic facts, and look at the status code before anything else.

In [ ]:
# Build params: action=query, titles=ARTICLE, prop=info, format=json
# Send the request with HEADERS, print the status code, stop if it isn't 200, then print the JSON
# Your code here

## Step 2 — Read the JSON one level at a time (5 minutes)

The answer is nested: `data["query"]["pages"]`, then a key that is the page's ID. We don't know the ID in advance, so take the first (and only) value. Print the title, the page ID, and the article's length in bytes.

In [ ]:
# Go down one level at a time: data -> "query" -> "pages" -> the one page inside
# Print its title, pageid, and length
# Your code here

## Step 3 — Every edit, one batch at a time (15 minutes)

`prop=revisions` returns an article's edits, newest first, at most 500 per request. When there are more, the response carries a `"continue"` key: add its contents to your parameters and ask again, until it's gone. Forgetting this loop is the most common API bug there is, because the first batch looks like a complete answer.

This step sends many requests, so first write `get(url, params)`, the one way every request from here on goes out. It always sends `HEADERS`, and when the answer is a 429 it waits as many seconds as `Retry-After` says and asks again, up to three times, as the rate-limit page asks.

Then write `get_revisions(title)`. Use `rvstart` to fix the newest edit you'll accept, `2026-09-30T23:59:59Z`, so that running it next week gives today's answer. Call `raise_for_status()` before `.json()`: if the rate limit still answers, you'll see why instead of a confusing JSON error.

In [ ]:
# Import time. Define get(url, params=None):
#   up to four tries: request url with params and HEADERS (timeout=30); return the response
#   unless it is a 429, in which case time.sleep() for int(response.headers["Retry-After"]) seconds
# Define get_revisions(title):
#   params: action=query, titles, prop=revisions, rvprop=ids|timestamp|user|size|comment,
#           rvlimit=max, rvstart="2026-09-30T23:59:59Z", format=json
#   loop: get(API, params), raise_for_status, keep this batch's revisions,
#         and if "continue" is in the response, add it to params; otherwise stop
#   return the revisions as a DataFrame
# Call it for ARTICLE and print how many revisions came back
# Your code here

## Step 4 — Chart the edits, and find the editors (5 minutes)

Turn `timestamp` into dates, count edits per month with `resample("ME")`, and plot them. Then use `value_counts()` on `user` to see who edits most, and what share of all edits the top ten made.

In [ ]:
# Convert timestamp to datetimes, count edits per month, and plot them
# Show the ten most active editors, and the share of all edits they made
# Your code here

## Step 5 — A second API for readers (5 minutes)

Pageviews come from a different API, Wikimedia's REST API, whose parameters are parts of the URL's path rather than a dictionary: project, access, **agent** (who is reading), article, granularity, start, and end. Define `get_pageviews()` in this cell, and use it in the next. Titles in the path use underscores, not spaces.

In [ ]:
# Define get_pageviews(title, start, end, agent="user", granularity="daily"):
#   build the URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article
#                  /en.wikipedia/all-access/{agent}/{article}/{granularity}/{start}/{end}
#   request it with get(), raise_for_status, make a DataFrame from ["items"]
#   turn "timestamp" (YYYYMMDDHH) into dates, and return the timestamp and views columns
# Your code here

## Step 6 — Plot the readers, and find the biggest day (8 minutes)

Get daily views from January 1, 2021 to September 30, 2026, plot them, and find the peak with `idxmax()`. What happened that day? The API's days are UTC days, so an evening event in Colorado lands on the next one.

In [ ]:
# Get daily views for ARTICLE from 20210101 to 20260930, and plot them
# Find the day with the most views, and print its date and count
# Your code here

## Step 7 — Who is reading? (7 minutes)

A pageview is not a person. Wikimedia splits its counts by agent: `user` (people), `spider` (crawlers that say what they are), and `automated` (bots that don't). Call `get_pageviews()` once for each, with monthly granularity, for 2024 to September 2026, and plot the three together. When Wikimedia improved its bot detection in 2025, it moved past traffic from one agent to another.

In [ ]:
# For each agent in user, spider, automated: get monthly views from 20240101 to 20260930
# Put them in one DataFrame, one column per agent (pivot), and plot it
# Your code here

## Step 8 — One function for everything else (10 minutes)

Links, categories, templates, languages, and what links here are all `action=query`, with a different `prop` or `list`, and they all paginate the same way. So write the loop once: `query_all(params)` follows `"continue"` and returns every batch. Then count the article's categories, its language editions, and the articles that link to it.

Look at the first few categories: some are **hidden** maintenance categories that readers never see. Add `clshow="!hidden"` to leave them out.

In [ ]:
# Define query_all(params): add action=query and format=json, then loop like Step 3,
#   collecting each response's "query" part; return the list of batches
# Use it to count: categories (prop=categories, cllimit=max), with and without clshow="!hidden";
#   language editions (prop=langlinks, lllimit=max);
#   articles linking here (list=backlinks, bltitle, bllimit=max, blnamespace=0)
# Your code here

## Step 9 — The same with a client library (5 minutes)

[mwclient](https://mwclient.readthedocs.io/) does the continuation for you. If you haven't installed it, run `%conda install -y -c conda-forge mwclient` in a new cell first; conda-forge has version 0.10.1. Ask it for the same article's categories, language editions, and revisions. In mwclient 0.10, `limit` is how many results to ask for in each request; version 0.11 calls it `api_chunk_size`.

Two things stay raw: mwclient speaks only the Action API, so pageviews still need Step 5, and a 429 from the rate limit still stops it.

In [ ]:
# Import mwclient and print its version
# Make a Site for en.wikipedia.org with clients_useragent=CLIENT (mwclient adds its own name and version)
# Get the page, then count its categories, its language editions,
#   and its revisions up to 2026-09-30T23:59:59Z, 500 per request (limit=500)
# Your code here

## Step 10 — What did we learn? (5 minutes)

In a few sentences, as comments or in a Markdown cell:

- Did the revision count from Step 3 match mwclient's in Step 9? What would it mean if it didn't?
- Does the biggest day in Step 6 line up with a burst of edits in Step 4?
- In Step 7, how much of the article's traffic is people? What would that mean for a study that used `all-agents`?
- Run Steps 1–8 again for an article you care about. What's different?

In [ ]:
# Your answers here, as comments
